# Lab 6: Your Project (optional)

**AI2Practice Research Lab** · about 4 to 6 hours, spread over as many sessions as you like · runs in Google Colab

## The question

**Your own question, on an open dataset, written up as a short abstract.**

This lab is different from the others. There is no single right answer. You choose a small question, plan how to answer it, run the analysis, write it up as a structured abstract of about 250 words, put it in a public GitHub repository, and ask a classmate to review it. The result is a portfolio piece: something concrete you can show a mentor when you ask to join a research project.

## What you will be able to do

By the end you will be able to turn a clinical curiosity into a question that data can answer, write an analysis plan before looking at the results, carry out the analysis with tools from Labs 1 to 5, write a structured abstract (Background, Methods, Results, Conclusion), and give and receive peer review through a pull request on GitHub.

## How this notebook works

Tasks are tagged with a **Mode** as in the other labs. **Write** means you write it yourself. **Reuse** means you use your own `toolkit.py` and the code from earlier labs. **Direct the AI** means you ask an AI assistant to write code, check it with a test cell, and fill in a short log. **Read** means you study an example before writing your own. The test cells here are lighter than in other labs, because your project is your own: they check that each step is done and in the right shape, not that you got a particular number.

Keep it small. A good first project answers one question with one dataset and one main number. Big ideas can come later.

## Set your GitHub username

Type your GitHub username between the quotes and run the cell. It is used to load your own versions of the Lab 1 to 3 datasets and your `toolkit.py`.

In [ ]:
GITHUB_USERNAME = "your-username"

## Setup

This cell downloads the course data generator (`rl_data.py`) and your `toolkit.py`, installs MedMNIST, and loads the usual libraries.

In [ ]:
import os, sys, subprocess, urllib.request

COURSE_RAW = "https://raw.githubusercontent.com/UM-IHC-CA2i/CA2i-Research-Launchpad/main/lesson-notebooks/"
FORK_RAW = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/CA2i-Research-Launchpad/main/lesson-notebooks/"

def fetch(filename, sources):
    if os.path.exists(filename):
        return
    for base in sources:
        try:
            urllib.request.urlretrieve(base + filename, filename)
            print("Downloaded", filename, "from", base)
            return
        except Exception:
            pass
    print("Could not download", filename)

fetch("rl_data.py", [COURSE_RAW])
fetch("toolkit.py", [FORK_RAW, COURSE_RAW])   # your own toolkit first

try:
    import medmnist
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "medmnist"])
    import medmnist
from medmnist import INFO

try:
    import rl_data
except Exception:
    rl_data = None
    print("rl_data.py is not available; the Lab 1-3 datasets will not load, but MedMNIST will.")

try:
    from toolkit import sens_spec
except Exception:
    def sens_spec(y_true, y_pred):
        """Return (sensitivity, specificity) for 0/1 labels."""
        pairs = list(zip(y_true, y_pred))
        tp = sum(1 for t, p in pairs if t == 1 and p == 1); fn = sum(1 for t, p in pairs if t == 1 and p == 0)
        tn = sum(1 for t, p in pairs if t == 0 and p == 0); fp = sum(1 for t, p in pairs if t == 0 and p == 1)
        return (tp / (tp + fn) if tp + fn else float("nan"), tn / (tn + fp) if tn + fp else float("nan"))
    print("Using a stand-in sens_spec (toolkit.py not found)")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("Ready.")

---
# Part 1: A question

A research question that data can answer names **who** or **what** you are studying, **what you measure**, and usually **what you compare**. "Is AI good at chest X-rays?" is too vague to answer. "On the PneumoniaMNIST test set, does a small CNN have lower specificity than sensitivity for pneumonia?" can be answered in an afternoon.

You can use any open dataset you like. Two easy starting points are already set up for you.

**MedMNIST** is a collection of small, labeled medical image datasets, all loaded the same way as PneumoniaMNIST in Lab 4. A few that suit a first project: `pneumoniamnist` (pediatric chest X-ray, normal vs pneumonia), `breastmnist` (breast ultrasound, 780 images, normal or benign vs malignant), `chestmnist` (adult chest X-ray, 14 findings that can appear together), `organamnist` (abdominal CT slices, 11 organs), `dermamnist` (skin photographs, 7 lesion types), and `bloodmnist` (blood cell microscopy, 8 cell types). Run the next cell to see the full list.

**Your own Lab 1 to 3 data**: the CT log from Lab 1, the clinical table from Lab 2 (remember to clean it first), and the pneumothorax detector output from Lab 3. These are synthetic (made up, but realistic), so they are good for practicing methods, not for real clinical claims.

In [ ]:
rows = []
for flag, info in INFO.items():
    if flag.endswith("3d"):
        continue   # skip the 3D datasets to keep things simple
    rows.append({"name": flag, "task": info["task"], "images": sum(info["n_samples"].values()),
                 "classes": len(info["label"])})
pd.DataFrame(rows)

### Task 1.1: Choose your data

**Mode: Reuse**

The helper below loads either a MedMNIST dataset (by its name, such as `"breastmnist"`) or one of your own Lab datasets (`"lab1"`, `"lab2"` or `"lab3"`). Set `DATA_SOURCE` and run the cell. For MedMNIST you get the three official splits, as in Lab 4; for Lab data you get a pandas table.

In [ ]:
def load_starter(name):
    name = name.lower().strip()
    if name in ("lab1", "lab2", "lab3"):
        if rl_data is None:
            raise RuntimeError("rl_data.py could not be downloaded; check your internet connection.")
        return rl_data.dataset(int(name[-1]), GITHUB_USERNAME)
    info = INFO[name]
    cls = getattr(medmnist, info["python_class"])
    return {split: cls(split=split, download=True) for split in ["train", "val", "test"]}

DATA_SOURCE = "___"     # for example "breastmnist" or "lab3"
data = load_starter(DATA_SOURCE)
if isinstance(data, dict):
    for split, ds in data.items():
        print(split, ds.imgs.shape)
    print("Labels:", INFO[DATA_SOURCE]["label"])
else:
    display(data.head())

In [ ]:
assert DATA_SOURCE != "___", "Set DATA_SOURCE to a dataset name."
assert data is not None and len(data) > 0
print("Check passed")

### Task 1.2: Write your question

**Mode: Write**

Write your question as one sentence ending with a question mark. Name the dataset, what you will measure, and what you will compare.

<details><summary>Examples</summary>

"On BreastMNIST ultrasound images, how sensitive and specific is a simple pixel-based model for malignancy, and does it beat always answering 'benign'?"

"In my Lab 1 CT log, do emergency department patients have a different body-region mix than outpatients?"

"In my Lab 3 detector data, does the best threshold for site A also work for site B?"
</details>

In [ ]:
QUESTION = ""

In [ ]:
assert QUESTION.strip().endswith("?"), "Write your question as a sentence ending with '?'."
assert len(QUESTION.split()) >= 8, "Make the question specific: name the data, the measure and the comparison."
print("Check passed")

---
# Part 2: A plan, then an analysis

## Plan before you look

Write down how you will answer the question **before** you see the results. If you try many analyses and report only the one that looks best, you will find "results" that are really chance. Writing the plan first protects you (and readers) from that. Real studies do the same by registering a protocol in advance.

### Task 2.1: Your analysis plan

**Mode: Write**

Fill in each entry of the `PLAN` dictionary below in a sentence or two. (A **dictionary** is a Python object that stores values under names, called keys.)

<details><summary>Example</summary>

```python
PLAN = {
    "data": "BreastMNIST, official train/val/test splits (546/78/156 images).",
    "outcome": "Label: malignant (0) vs normal or benign (1); I will treat malignant as the positive class.",
    "method": "Logistic regression on the 784 pixel values, trained on the training set only.",
    "main_measure": "Sensitivity and specificity for malignancy on the test set.",
    "comparison": "A baseline that always answers 'not malignant'.",
}
```
</details>

In [ ]:
PLAN = {
    "data": "",          # which dataset and which rows or splits
    "outcome": "",       # what you are predicting or describing
    "method": "",        # what you will compute or train
    "main_measure": "",  # the one number that answers your question
    "comparison": "",    # what you compare it to (another group, a baseline, a threshold)
}

In [ ]:
for key in ["data", "outcome", "method", "main_measure", "comparison"]:
    assert len(PLAN.get(key, "").split()) >= 3, f"Write at least a short sentence for '{key}'."
print("Check passed")

## Run the analysis

### Task 2.2: The main analysis

**Mode: Direct the AI**

Ask an AI assistant to write the code that carries out your plan. Give it your `PLAN`, the name of your data variable (`data`), and what it contains (for MedMNIST: a dictionary of splits, each with `.imgs` and `.labels`; for Lab data: a pandas table and its column names). Ask it to put your main results in a dictionary called `results` that maps names to numbers, for example `{"sensitivity": 0.81, "specificity": 0.64}`. Reuse your own functions where they fit, such as `sens_spec` from your toolkit; tell the AI they exist.

Read the code before you run it. Does it train only on the training data and evaluate only on the test data? Does it use the positive class you chose? Those are the mistakes you fixed in earlier labs, and AI assistants still make them.

<details><summary>Example for the BreastMNIST plan</summary>

```python
from sklearn.linear_model import LogisticRegression

def xy(split):
    X = data[split].imgs.reshape(len(data[split].imgs), -1) / 255.0
    y = (data[split].labels.ravel() == 0).astype(int)    # 1 = malignant
    return X, y

X_tr, y_tr = xy("train"); X_te, y_te = xy("test")
clf = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
y_hat = clf.predict(X_te)
sens, spec = sens_spec(y_te, y_hat)
base_sens, base_spec = sens_spec(y_te, np.zeros_like(y_te))
results = {"test_images": len(y_te), "malignant_in_test": int(y_te.sum()),
           "sensitivity": sens, "specificity": spec,
           "baseline_sensitivity": base_sens, "baseline_specificity": base_spec}
results
```
</details>

In [ ]:
# Paste the AI's code here (and edit it as needed)

In [ ]:
assert isinstance(results, dict) and len(results) >= 1, "Store your main numbers in a dictionary called results."
for k, v in results.items():
    assert isinstance(v, (int, float, np.integer, np.floating)), f"results['{k}'] should be a number."
print("Check passed")
for k, v in results.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, (float, np.floating)) else f"  {k}: {v}")

**AI-use log, Task 2.2.** Double-click this cell and fill it in. If you publish this project, a log like this lets you describe honestly how AI was used, which many journals now ask for.

*Prompt I used:* (paste your prompt here)

*What I changed in the AI's code, and why:* (write "nothing" if you changed nothing)

*Did the test cell pass on the first try? If not, what went wrong?*

### Task 2.3: One figure

**Mode: Direct the AI**

Most abstracts can carry one figure or table, and a good figure often explains the result faster than words. Ask an AI assistant for code that draws one clear figure of your main result, with labeled axes and a title, and saves it as `figure1.png`. Check it the way a reviewer would: are the axes labeled, are the units clear, and could someone understand it without reading your code?

<details><summary>Example</summary>

```python
fig, ax = plt.subplots(figsize=(5, 3.5))
names = ["Model", "Always 'benign'"]
ax.bar([n + "
sensitivity" for n in names], [results["sensitivity"], results["baseline_sensitivity"]])
ax.bar([n + "
specificity" for n in names], [results["specificity"], results["baseline_specificity"]])
ax.set_ylim(0, 1); ax.set_ylabel("Proportion"); ax.set_title("BreastMNIST test set (n = 156)")
plt.tight_layout(); fig.savefig("figure1.png", dpi=200); plt.show()
```
</details>

In [ ]:
# Paste the AI's code here

In [ ]:
assert os.path.exists("figure1.png"), "Save your figure with fig.savefig('figure1.png')."
print("Check passed")

**AI-use log, Task 2.3.** Double-click this cell and fill it in. If you publish this project, a log like this lets you describe honestly how AI was used, which many journals now ask for.

*Prompt I used:* (paste your prompt here)

*What I changed in the AI's code, and why:* (write "nothing" if you changed nothing)

*Did the test cell pass on the first try? If not, what went wrong?*

### Task 2.4: Your own check

**Mode: Reuse**

Every result deserves at least one check that does not depend on the AI's code. Pick one number from `results` and recompute it a different way, using your own code or your `toolkit.py` (for example, count true positives by hand with `(y_true == 1) & (y_pred == 1)`, or recount the rows in a group). Store the recomputed value in `my_check` and the name of the result it checks in `checked_key`.

<details><summary>Example</summary>

```python
tp = ((y_te == 1) & (y_hat == 1)).sum()
my_check = tp / (y_te == 1).sum()
checked_key = "sensitivity"
```
</details>

In [ ]:
my_check = None
checked_key = ""

In [ ]:
assert checked_key in results, "checked_key should be one of the names in results."
assert my_check is not None and abs(float(my_check) - float(results[checked_key])) < 1e-6, \
    "Your recomputed value does not match. Find out which one is wrong before you write it up."
print("Check passed")

---
# Part 3: A structured abstract

An **abstract** is the short summary at the top of a research paper or conference submission, and often the only part people read. Medical journals and meetings (such as RSNA or the AUR annual meeting) ask for a **structured** abstract with fixed headings, usually Background (or Purpose), Methods, Results and Conclusion, in about 250 to 300 words.

### Task 3.1: Read an example

**Mode: Read**

Read this example, based on the Lab 4 model. As you read, notice three things: the Background ends with the question; the Results give actual numbers with the number of images; and the Conclusion does not claim more than the data show.

> **Background.** Small convolutional neural networks (CNNs) can classify chest X-rays, but accuracy alone can hide poor specificity and sensitivity to changes in image appearance. We asked how a small CNN trained on pediatric chest X-rays performs on an independent test set and on the same images after a simulated change of scanner.
>
> **Methods.** We used PneumoniaMNIST, a public set of 5,856 pediatric chest X-rays downsampled to 28 by 28 pixels, with its official training (4,708), validation (524) and test (624; 390 pneumonia) splits. A two-layer CNN was trained for five epochs. We measured accuracy, sensitivity and specificity for pneumonia on the test set, and repeated the measurement after a simulated second hospital (brightness increased, contrast reduced). We also trained a model on images in which a corner marker appeared only on pneumonia cases, to illustrate shortcut learning, and inspected gradient saliency maps.
>
> **Results.** On the test set, accuracy was 81%, with sensitivity 99% and specificity 52%. Under the simulated second hospital, accuracy fell to 64% and specificity to 3%, with nearly all images called pneumonia. The marker-trained model reached 98% accuracy when the marker was present and 46% without it; its saliency maps highlighted the marker rather than the lungs.
>
> **Conclusion.** A small CNN detected most pneumonia cases but over-called normal X-rays, and its performance depended heavily on image appearance. Low-resolution public datasets are useful for teaching and method development, but external validation and saliency checks are needed before any clinical claim.

(The numbers are from one training run; yours will differ.)

Quick check on your reading: according to the example, how many test images were there, and what was the specificity under the simulated second hospital (as a percentage)? Fill in the two numbers.

In [ ]:
example_test_images = ___
example_spec_second_hospital = ___

In [ ]:
assert example_test_images == 624, "Look at the Methods section: the test split size is given in brackets."
assert example_spec_second_hospital == 3, "Look at the Results section, second sentence."
print("Check passed")
print("Notice that both numbers come with enough context to judge them: a denominator and the condition they were measured under.")

### Task 3.2: Write your abstract

**Mode: Write**

Write your own abstract between the triple quotes, with the four headings **Background**, **Methods**, **Results** and **Conclusion**, in about 250 words (the test accepts 150 to 300). Write it yourself first. Afterwards, you may ask an AI assistant to point out what is unclear or overstated (not to rewrite it), and note in the log below what you changed. Then the abstract stays yours.

A few rules of thumb. Give numbers with their denominators ("390 of 624"). Say what kind of data it is (synthetic, public, downsampled). Do not write "the model can diagnose pneumonia" when you tested it on 28-pixel images from one source; write what you measured.

In [ ]:
ABSTRACT = """
Background:

Methods:

Results:

Conclusion:
"""

In [ ]:
import re
words = len(ABSTRACT.split())
for heading in ["Background", "Methods", "Results", "Conclusion"]:
    assert re.search(heading, ABSTRACT, re.IGNORECASE), f"Your abstract needs a '{heading}' heading."
assert 150 <= words <= 300, f"Your abstract has {words} words; aim for about 250 (150 to 300)."
print("Check passed")
print("Word count:", words)

**AI-use log, Task 3.2 (only if you asked an AI for feedback).**

*Prompt I used:*

*What the AI pointed out, and what I changed:*

---
# Part 4: Share it and get a peer review

## Your project repository

A **repository** (repo) is a folder of files on GitHub, with its full history of changes. Make a new **public** repository for this project so a mentor can see it.

1. On GitHub, click **+ > New repository**. Give it a short name, such as `breastmnist-baseline`. Choose **Public**, tick **Add a README file**, and click **Create repository**.
2. Back in Colab, click **File > Save a copy in GitHub**, choose your new repository, and click **OK**. Your notebook is now in the repo.
3. Upload `figure1.png` (in Colab, find it in the Files panel on the left, download it, then on GitHub click **Add file > Upload files**).
4. Edit the `README.md` on GitHub (the pencil icon): paste your question and your abstract, and add a line saying which dataset you used and that the code was written with help from an AI assistant where your logs say so.

## Peer review by pull request

A **pull request** (PR) is how people suggest changes to someone else's repository on GitHub. The reviewer makes changes in their own copy and then "requests" that the owner "pull" those changes in. The owner sees exactly which lines would change, can discuss each one in comments, and decides whether to accept (**merge**) them. Almost all collaborative code and many papers are reviewed this way.

**To review a classmate's project:**

1. Open their repository and click **Fork** to make your own copy.
2. In your copy, click **Add file > Create new file** and name it `REVIEW.md`. Write a short review: Is the question clear? Do the Methods match what the notebook actually does? Are the numbers in the abstract the same as in the notebook output? Is the Conclusion supported? Name one thing that works well and one thing to improve. You can also edit a typo directly in their README.
3. Click **Commit changes**, then **Contribute > Open pull request**. Give it a title such as "Peer review from your-username" and click **Create pull request**.

**When you receive a review:** read the PR, reply to the comments in the **Conversation** tab, make any changes you agree with, and click **Merge pull request** to keep the review in your repo's history. A mentor looking at your repo will see that your work was reviewed and how you responded.

No classmate yet? Post your repo link in the AI2Practice community or ask a friend who codes. You can also review your own work after a day away, using the same questions.

---
# Report back

Put the address of your public project repository below and run the cell. Then copy the URL and your abstract into the Lab 6 page of the AI2Practice site.

In [ ]:
REPO_URL = "https://github.com/your-username/your-repo"

In [ ]:
import re
assert re.fullmatch(r"https://github\.com/[A-Za-z0-9-]+/[A-Za-z0-9._-]+/?", REPO_URL.strip()), \
    "The URL should look like https://github.com/<user>/<repo>"
assert "your-username" not in REPO_URL, "Replace the example with your own repository URL."
print("Check passed")
print()
print("Repository:", REPO_URL.strip())
print()
print(ABSTRACT.strip())

# Save to GitHub

Your project notebook should live in your **project repository** (Part 4, step 2). If you change anything after that, save again: **File > Save a copy in GitHub**, choose the same repository and file name, write a short commit message such as "Update abstract", and click **OK**.

If you would also like a copy in your fork of the course repository (`your-username/CA2i-Research-Launchpad`, your copy of `https://github.com/UM-IHC-CA2i/CA2i-Research-Launchpad`), save once more there with the file path `lesson-notebooks/Lab_6_Your_Project.ipynb`.

Congratulations on finishing the Research Lab. Your repo, your abstract and your review history are a solid start for a conversation with a research mentor; the Getting Involved section of the Research Lab page has tips on finding one.